<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">

# Generating Synthetic Data With NVIDIA Cosmos™ Predict

## Using Cosmos for Synthetic Data Generation in Digital Twins

[NVIDIA Cosmos™](https://www.nvidia.com/en-us/ai/cosmos/) is a platform of state-of-the-art generative world foundation models (WFMs), advanced tokenizers, guardrails, and an accelerated data processing and curation pipeline. It is built to power world model training and accelerate physical AI development for autonomous vehicles (AVs) and robots.

In this notebook, we will run inference using Cosmos Predict to generate synthetic data. Cosmos Predict is a model that generates future world states in the form of photorealistic videos from text, image, or video input. This helps generate synthetic data based on user input. This data can be used in addition to the data we create using Cosmos Transfer (seen previously) to add further visual diversity, but without the requirement of ground truth data. Ultimately, we use the whole dataset for fine-tuning to train VSS (Video Search and Summarization) or VLM (Vision Language) models to identify various objects in different environments

### Learning Objective
In this notebook, we will focus on: 
- Generating synthetic data with NVIDIA Cosmos to power physical AI workflows

### Table of Contents

**[Set Up the Environment](#Set-Up-the-Environment)** <br>
**[Understand Cosmos Predict Models](#Understand-Cosmos-Predict-Models)** <br>
**[Generate Synthetic Data Using Cosmos Predict](#Generate-Synthetic-Data-Using-Cosmos-Predict)** <br>
**[Use the Factory Data for World Generation](#Use-the-Factory-Data-for-World-Generation)** <br>
**[Scale Synthetic Data Generation Through Batch Processes](#Scale-Synthetic-Data-Generation-Through-Batch-Processes)** <br>
**[Review](#Review)** <br>

---
### Set Up the Environment
Firts we check that the Cosmos models are downloaded and available.

In [1]:
import os
import time

# Check for download complete every 30 seconds
while not os.path.isfile("/dli/task/Cosmos/Cosmos/checkpoints/download_complete"):
    print("Download still in progress. Checking again in 30 seconds...")
    time.sleep(30)
print("Download is complete.")

Download is complete.


We run the next two cells to set some environment variables.

In [4]:
import os
import subprocess
from IPython.display import HTML

# Prepare environment variables
env = os.environ.copy()
env["PYTHONPATH"] = os.getcwd() + "/Cosmos/"
env["CUDA_VISIBLE_DEVICES"] = "0"

In [5]:
working_directory = "/dli/task/Cosmos/Cosmos"

---
### Understand Cosmos Predict Models

We will be running some of the Cosmos Predict models for synthetic data generation. These models are used for future state prediction, and they generate visual simulations based on text prompts and video prompts.

<img src="images/predict1_diagram.png" alt="predict1 diagram" width=800>

Cosmos Predict includes diffusion-based and autoregressive-based world foundation models for Text2World and Video2World generation. This can be used for synthetic data generation for training AI models.

The Text2World model takes input in the form of a text prompt and generates a video from it. This is useful to generate videos of varied environments with different things happening in them. For example, in our factory use-case, you can describe the factory environment, what objects you want in it, and what events are occurring in the form of a text prompt. Eventually, these videos can be used to train VSS (Video Search and Summarization) or VLM (Vision Language) models, which are deployed in real factories to identify events happening on the factory floor. 

The Video2World model takes input videos and generates future frames of the video. It helps generate future scenarios of what may happen in the given scene and videos generated in this way can also be used to train vision AI models like VSS.

Omniverse was used to generate the input video to the Video2World model. In the workshop, we saw how to capture videos of our digital twin in Omniverse. These videos of the digital twin allow us to guide the inference process to generate output videos of our factory according to our requirements. Once we have the digital twin of our factory set up in Omniverse, it is quick, easy and cost-effective to generate videos of future scenarios using Cosmos Predict. These scenarios are very expensive and dangerous to set up and capture in the real factory.    


<img src="images/nvidia-cosmos-synthetic-data-generation-ari.jpeg" width=1000 alt="OV and Cosmos for data generation">

---
### Generate Synthetic Data Using Cosmos Predict

We will now run the Text2World model. This model takes input in the form of a text prompt and generates a photorealistic video. This is useful to generate videos of varied environments with different things happening in them. These videos can be used to train VSS models, which are deployed in real factories to identify events happening on the factory floor. We will see an example of VSS in the next part of this workshop.

In the next cell, we set the text prompt that will be used to generate a video. Guardrail models are built-in to ensure safety of prompts and generated output. 

In [8]:
# Parameters
prompt =  """
        A busy bridge inspection scene: engineers in reflective vests walk along the bridge, using drones to inspect structural elements.
        Vehicles pass in the background while sensors scan the surface for cracks or rust. The sky is clear, shadows move across the bridge deck.
        The environment is realistic with photorealistic textures, dynamic movement, and varied lighting throughout the video. """



video_save_name = "Cosmos-1.0-Diffusion-7B-Text2World"

In the next cell, we execute the command to run inference using Cosmos Text2World models. Run the next cell and wait for a few minutes for video generation.

In [9]:
# Inference
command = [
    "python", "cosmos1/models/diffusion/inference/text2world.py",
    "--checkpoint_dir", "checkpoints",
    "--diffusion_transformer_dir", "Cosmos-1.0-Diffusion-7B-Text2World",
    "--prompt", prompt,
    "--offload_prompt_upsampler",
    "--video_save_name", video_save_name
]

subprocess.run(command, env=env, cwd=working_directory)

/usr/local/lib/python3.10/dist-packages/transformer_engine/pytorch/attention.py:108: UserWarning: To use flash-attn v3, please use the following commands to install: 
(1) pip install "git+https://github.com/Dao-AILab/flash-attention.git#egg=flashattn-hopper&subdirectory=hopper" 
(2) python_path=`python -c "import site; print(site.getsitepackages()[0])"` 
(3) mkdir -p $python_path/flashattn_hopper 
(4) wget -P $python_path/flashattn_hopper https://raw.githubusercontent.com/Dao-AILab/flash-attention/main/hopper/flash_attn_interface.py
  warnings.warn(


[09-27 15:22:24|INFO|cosmos1/utils/misc.py:106:set_random_seed] Using random seed 1.


Loading checkpoint shards: 100%|██████████| 3/3 [00:03<00:00,  1.02s/it]
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/usr/local/lib/python3.10/dist-packages/torch/serialization.py:1243: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  warnings.warn(


[09-27 15:23:27|INFO|cosmos1/models/diffusion/inference/world_generation_pipeline.py:314:generate] Run with prompt: 
        A busy bridge inspection scene: engineers in reflective vests walk along the bridge, using drones to inspect structural elements.
        Vehicles pass in the background while sensors scan the surface for cracks or rust. The sky is clear, shadows move across the bridge deck.
        The environment is realistic with photorealistic textures, dynamic movement, and varied lighting throughout the video. 
[09-27 15:23:27|INFO|cosmos1/models/diffusion/inference/world_generation_pipeline.py:315:generate] Run with negative prompt: The video captures a series of frames showing ugly scenes, static with no motion, motion blur, over-saturation, shaky footage, low resolution, grainy texture, pixelated images, poorly lit areas, underexposed and overexposed scenes, poor color balance, washed out colors, choppy sequences, jerky movements, low frame rate, artifacting, color bandi

CompletedProcess(args=['python', 'cosmos1/models/diffusion/inference/text2world.py', '--checkpoint_dir', 'checkpoints', '--diffusion_transformer_dir', 'Cosmos-1.0-Diffusion-7B-Text2World', '--prompt', '\n        A busy bridge inspection scene: engineers in reflective vests walk along the bridge, using drones to inspect structural elements.\n        Vehicles pass in the background while sensors scan the surface for cracks or rust. The sky is clear, shadows move across the bridge deck.\n        The environment is realistic with photorealistic textures, dynamic movement, and varied lighting throughout the video. ', '--offload_prompt_upsampler', '--video_save_name', 'Cosmos-1.0-Diffusion-7B-Text2World'], returncode=0)

Run the next cell to see the generated video output.

In [10]:
video_path = "Cosmos/outputs/" + video_save_name + ".mp4"
HTML(f"""
    <video width="640" height="480" controls>
        <source src="{video_path}" type="video/mp4">
    </video>
""")

---
### Use the Factory Data for World Generation

We will now run the Video2World model. This model takes input videos and generates future frames of the video. The 'future-scenario' videos generated in this way can be used to train vision AI models to identify specific events in videos. 

In the next cell, we set the input video that will be used by Cosmos to generate future frames.

In [11]:
# Parameters
input_image_or_video_path = "cosmos1/models/autoregressive/assets/v1p0/factory_input.mp4"
video_save_name = "Cosmos-1.0-Autoregressive-4B"
top_p = 0.8
temperature = 1.0

Run the next cell to see the video that we are giving as input to the Cosmos model.

In [12]:
video_path = "Cosmos/cosmos1/models/autoregressive/assets/v1p0/factory_input.mp4"
HTML(f"""
    <video width="640" height="480" controls>
        <source src="{video_path}" type="video/mp4">
    </video>
""")

In the next cell, we execute the command to run inference using Cosmos. Run the next cell and wait for a few minutes for video generation.

In [13]:
# Run inference
command = [
    "python", "cosmos1/models/autoregressive/inference/base.py",
    "--input_type", "video",
    "--input_image_or_video_path", input_image_or_video_path,
    "--video_save_name", video_save_name,
    "--ar_model_dir", "Cosmos-1.0-Autoregressive-4B",
    "--top_p", str(top_p),
    "--temperature", str(temperature),
]

subprocess.run(command, env=env, check=True, cwd=working_directory)

/usr/local/lib/python3.10/dist-packages/transformer_engine/pytorch/attention.py:108: UserWarning: To use flash-attn v3, please use the following commands to install: 
(1) pip install "git+https://github.com/Dao-AILab/flash-attention.git#egg=flashattn-hopper&subdirectory=hopper" 
(2) python_path=`python -c "import site; print(site.getsitepackages()[0])"` 
(3) mkdir -p $python_path/flashattn_hopper 
(4) wget -P $python_path/flashattn_hopper https://raw.githubusercontent.com/Dao-AILab/flash-attention/main/hopper/flash_attn_interface.py
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/serialization.py:1243: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use '

[09-27 15:28:16|INFO|cosmos1/models/autoregressive/inference/base.py:91:main] Run with image or video path: factory_input.mp4
[09-27 15:28:16|INFO|cosmos1/models/autoregressive/inference/world_generation_pipeline.py:411:generate] Run generation
[09-27 15:28:16|INFO|cosmos1/models/autoregressive/inference/world_generation_pipeline.py:310:_run_model_with_offload] Using input size of 9 frames
[09-27 15:28:17|INFO|cosmos1/utils/misc.py:106:set_random_seed] Using random seed 0.
[09-27 15:28:17|INFO|cosmos1/models/autoregressive/model.py:380:generate] Compiled AR sampling function. Note: the first run will be slower due to compilation


/usr/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


[09-27 15:30:03|INFO|cosmos1/models/autoregressive/inference/world_generation_pipeline.py:415:generate] Finish AR model generation
[09-27 15:30:03|INFO|cosmos1/models/autoregressive/inference/world_generation_pipeline.py:418:generate] Run diffusion decoder on generated tokens
[09-27 15:30:45|INFO|cosmos1/models/autoregressive/inference/world_generation_pipeline.py:422:generate] Finish diffusion decoder on generated tokens
[09-27 15:30:45|INFO|cosmos1/models/autoregressive/inference/world_generation_pipeline.py:426:generate] Run guardrail on generated video
[09-27 15:30:46|INFO|cosmos1/models/autoregressive/inference/world_generation_pipeline.py:431:generate] Finish guardrail on generated video
[09-27 15:30:46|INFO|cosmos1/models/autoregressive/inference/base.py:110:main] Saved video to outputs/Cosmos-1.0-Autoregressive-4B.mp4


CompletedProcess(args=['python', 'cosmos1/models/autoregressive/inference/base.py', '--input_type', 'video', '--input_image_or_video_path', 'cosmos1/models/autoregressive/assets/v1p0/factory_input.mp4', '--video_save_name', 'Cosmos-1.0-Autoregressive-4B', '--ar_model_dir', 'Cosmos-1.0-Autoregressive-4B', '--top_p', '0.8', '--temperature', '1.0'], returncode=0)


Run the next cell to see the generated video output. Compare it with the input video we had used and observe that coherent future frames are generated.

In [14]:
video_path = "Cosmos/outputs/" + video_save_name + ".mp4"
HTML(f"""
    <video width="640" height="480" controls>
        <source src="{video_path}" type="video/mp4">
    </video>
""")

---
### Scale Synthetic Data Generation Through Batch Processes

Let's now run an example of **batch generation** using the Video2World model we saw above. This example shows how to scale up the process of data generation by running inference on a batch of prompts, provided through the `batch_input_path` argument. This argument is the path to a JSONL file, which contains one visual input per line like this:

```json
{"visual_input": "path/to/video1.mp4"}
{"visual_input": "path/to/video2.mp4"}
```

Run the following cell to see the 6 videos that we will use as input for batch inference.

In [15]:
import os
from IPython.display import display, HTML

input_videos_path = "Cosmos/cosmos1/models/autoregressive/assets/v1p0/factory_batch_inputs"
input_video_files = [f for f in os.listdir(input_videos_path) if f.endswith('.mp4')]
video_tags = [
    f'<video src="{os.path.join(input_videos_path, file)}" width="320" controls></video>'
    for file in input_video_files
]
html = "".join(video_tags)
display(HTML(html))


In [16]:
# Run this cell to set the JSONL file with input videos and output folder path where videos will be saved.

batch_input_videos = "cosmos1/models/autoregressive/assets/v1p0/factory_batch_inputs/base.jsonl"
video_save_folder = "Cosmos-1.0-Autoregressive-4B-batch"

The Cosmos model uses the `top_p` and `temperature` parameters for generating output videos. These parameters can be changed to generate different videos. We have used the following values for optimal performance. However, these parameters can be tuned programmatically during batch inference to generate multiple output videos from a single input video.

In [ ]:
# Run the following cell to set parameter values

top_p = 0.8
temperature = 1.0

In [ ]:
# Run batch inference
command = [
    "python", "cosmos1/models/autoregressive/inference/base.py",
    "--input_type", "video",
    "--batch_input_path", batch_input_videos,
    "--video_save_folder", video_save_folder,
    "--ar_model_dir", "Cosmos-1.0-Autoregressive-4B",
    "--top_p", str(top_p),
    "--temperature", str(temperature),
]

subprocess.run(command, env=env, check=True, cwd=working_directory)

/usr/local/lib/python3.10/dist-packages/transformer_engine/pytorch/attention.py:108: UserWarning: To use flash-attn v3, please use the following commands to install: 
(1) pip install "git+https://github.com/Dao-AILab/flash-attention.git#egg=flashattn-hopper&subdirectory=hopper" 
(2) python_path=`python -c "import site; print(site.getsitepackages()[0])"` 
(3) mkdir -p $python_path/flashattn_hopper 
(4) wget -P $python_path/flashattn_hopper https://raw.githubusercontent.com/Dao-AILab/flash-attention/main/hopper/flash_attn_interface.py
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/serialization.py:1243: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use '

[09-27 15:31:27|INFO|cosmos1/models/autoregressive/utils/inference.py:327:load_vision_input] Reading batch inputs from path: cosmos1/models/autoregressive/assets/v1p0/factory_batch_inputs/base.jsonl
[09-27 15:32:48|INFO|cosmos1/models/autoregressive/inference/base.py:91:main] Run with image or video path: 0.mp4
[09-27 15:32:48|INFO|cosmos1/models/autoregressive/inference/world_generation_pipeline.py:411:generate] Run generation
[09-27 15:32:48|INFO|cosmos1/models/autoregressive/inference/world_generation_pipeline.py:310:_run_model_with_offload] Using input size of 9 frames
[09-27 15:32:49|INFO|cosmos1/utils/misc.py:106:set_random_seed] Using random seed 0.
[09-27 15:32:49|INFO|cosmos1/models/autoregressive/model.py:380:generate] Compiled AR sampling function. Note: the first run will be slower due to compilation


/usr/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


[09-27 15:33:31|INFO|cosmos1/models/autoregressive/inference/world_generation_pipeline.py:415:generate] Finish AR model generation
[09-27 15:33:31|INFO|cosmos1/models/autoregressive/inference/world_generation_pipeline.py:418:generate] Run diffusion decoder on generated tokens
[09-27 15:34:12|INFO|cosmos1/models/autoregressive/inference/world_generation_pipeline.py:422:generate] Finish diffusion decoder on generated tokens
[09-27 15:34:12|INFO|cosmos1/models/autoregressive/inference/world_generation_pipeline.py:426:generate] Run guardrail on generated video
[09-27 15:34:14|INFO|cosmos1/models/autoregressive/inference/world_generation_pipeline.py:431:generate] Finish guardrail on generated video
[09-27 15:34:14|INFO|cosmos1/models/autoregressive/inference/base.py:110:main] Saved video to Cosmos-1.0-Autoregressive-4B-batch/0.mp4
[09-27 15:34:14|INFO|cosmos1/models/autoregressive/inference/base.py:91:main] Run with image or video path: 1.mp4
[09-27 15:34:14|INFO|cosmos1/models/autoregressi

Run the following cell to see the six output videos generated, one corresponding to each input video in the batch.

In [ ]:
import os
from IPython.display import display, HTML

output_videos_path = "Cosmos/Cosmos-1.0-Autoregressive-4B-batch"

output_video_files = [f for f in os.listdir(output_videos_path) if f.endswith('.mp4')]
video_tags = [
    f'<video src="{os.path.join(output_videos_path, file)}" width="320" controls></video>'
    for file in output_video_files
]
html = "".join(video_tags)
display(HTML(html))

---
### Review

In this notebook you learned the following:
- How to generate a photorealistic video using text prompts with Cosmos Predict
- How to generate a photorealistic video using another input video with Cosmos Predict
- How to scale up video generation wit tuning parameters to get different videos from the same input video

These generated photorealistic videos are used to train vision AI models like VSS, which are used in factories to identify events like failures or hazardous situations. In the next part of the workshop, we will learn more about VSS!